In [ ]:
import os, sys, glob, json, subprocess
import site
site.addsitedir('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/')
import torch
print('CUDA:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU{i}: {p.name} | {p.total_memory/1024**3:.1f} GB | sm_{p.major}{p.minor}')
# locate train.csv (competition mount path varies)
cands = glob.glob('/kaggle/input/*/train.csv')
print('train.csv candidates:', cands)
import csv
train_data = []
if cands:
    with open(cands[0], encoding='utf-8') as f:
        for row in csv.DictReader(f):
            q = row.get('prompt') or row.get('question') or ''
            a = row.get('answer') or row.get('completion') or ''
            if q and a: train_data.append({'prompt': q, 'completion': str(a)})
            if len(train_data) >= 50: break
print('loaded', len(train_data), 'training rows')

In [ ]:
import kagglehub
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- HARD GPU GUARD: P100 (sm_60) is incompatible with the PyTorch wheel ----
props = torch.cuda.get_device_properties(0)
cc = props.major + props.minor/10
vram_gb = props.total_memory/1024**3
print(f'GPU: {props.name} | {vram_gb:.1f} GB | sm_{props.major}{props.minor}')
assert cc >= 7.0, (
    f'GPU {props.name} (sm_{props.major}{props.minor}) is INCOMPATIBLE. '
    'Switch Accelerator to NVIDIA RTX Pro 6000 or GPU T4 x2 (NOT P100) and re-run.')

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
OUTPUT_DIR = '/kaggle/working'
LORA_RANK = 32

# ---- VRAM-ADAPTIVE LOAD: bf16 if big GPU, else 4-bit NF4 ----
if vram_gb >= 70:
    print('Loading full bf16 (large GPU)')
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map='auto', trust_remote_code=True, dtype=torch.bfloat16)
else:
    print('Loading 4-bit NF4 (<70GB GPU)')
    import subprocess, sys
    try:
        import bitsandbytes  # noqa
    except ImportError:
        subprocess.run([sys.executable,'-m','pip','install','-q','bitsandbytes'], check=False)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map='auto', trust_remote_code=True,
                                                 quantization_config=bnb, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('Model loaded.')

from peft import prepare_model_for_kbit_training
if vram_gb < 70:
    model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=LORA_RANK, lora_alpha=16,
    target_modules=r'.*\.(in_proj|out_proj|up_proj|down_proj)$',
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---- TRAINING (1 epoch, answer-masked CE) ----
if train_data:
    MAX_SEQ, GA = 2048, 8
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    dev = next(model.parameters()).device
    def datum(p, a):
        pt = tokenizer(p, add_special_tokens=False)['input_ids']
        at = tokenizer(a, add_special_tokens=False)['input_ids'] + [tokenizer.eos_token_id]
        t = (pt + at)[:MAX_SEQ]; m = ([0]*len(pt) + [1]*len(at))[:MAX_SEQ]
        return torch.tensor(t).unsqueeze(0), torch.tensor(m).unsqueeze(0)
    model.train(); step=0; rl=0.0
    for d in train_data:
        ids, mask = datum(d['prompt'], d['completion']); ids, mask = ids.to(dev), mask.to(dev)
        lg = model(input_ids=ids).logits
        sl, slb, sm = lg[..., :-1, :].contiguous(), ids[..., 1:].contiguous(), mask[..., 1:].contiguous()
        lf = torch.nn.CrossEntropyLoss(reduction='none')
        loss = (lf(sl.view(-1, sl.size(-1)), slb.view(-1)) * sm.view(-1)).sum() / sm.sum().clamp(min=1)
        (loss / GA).backward(); rl += loss.item()
        if (step+1) % GA == 0: opt.step(); opt.zero_grad()
        step += 1
        if step % 10 == 0: print(f'  step {step}/{len(train_data)} loss={rl/10:.4f}'); rl=0.0
    print('Training done.')

print('Saving adapter...')
model.save_pretrained(OUTPUT_DIR)
cfg = json.load(open(os.path.join(OUTPUT_DIR,'adapter_config.json')))
assert cfg.get('r',999) <= 32
print('adapter r =', cfg.get('r'))

In [ ]:
def boxed(t):
    if not t: return None
    i = t.rfind(chr(92) + 'boxed{')
    if i < 0: return None
    s = i + 7; d = 1
    for j in range(s, len(t)):
        if t[j] == '{': d += 1
        elif t[j] == '}':
            d -= 1
            if d == 0: return t[s:j]
    return None
def ok(pr, go):
    p = boxed(pr)
    if p is None: return False
    g = boxed(go) or str(go).strip(); p = p.strip()
    if p == g.strip(): return True
    try: return abs(float(p) - float(g)) <= 1e-2
    except: return False
preds = []
if train_data:
    model.eval(); dev = next(model.parameters()).device
    with torch.no_grad():
        for d in train_data[:10]:
            inp = tokenizer(d['prompt'], return_tensors='pt').to(dev)
            o = model.generate(**inp, max_new_tokens=256, do_sample=False)
            txt = tokenizer.decode(o[0], skip_special_tokens=True)[len(d['prompt']):].strip()
            preds.append(ok(txt, d['completion']))
sc = (sum(preds)/len(preds)) if preds else 0.0
json.dump({'score': sc, 'n': len(preds), 'correct': sum(preds)}, open('/kaggle/working/cv_score.json', 'w'))
print(f'=== CV {sc:.4f} ({sum(preds)}/{len(preds)}) ===')

In [ ]:
import glob as _g
os.chdir('/kaggle/working')
wfiles = [os.path.basename(p) for p in _g.glob('/kaggle/working/adapter_model*')]
subprocess.run('zip -j submission.zip adapter_config.json ' + ' '.join(wfiles), shell=True)
print('submission.zip exists:', os.path.exists('/kaggle/working/submission.zip'))
subprocess.run('unzip -l submission.zip', shell=True)